# Train your cloned voice on a free Colab GPU

1. On the Mac: `bash distill/run_all.sh <voice>` makes the datasets (or `make_dataset.py` only), then `bash distill/pack_for_colab.sh <voice>`.
2. Here: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
3. Download `piper_<voice>.zip` and unzip it into the project folder (it contains `voices/<voice>/piper/*.onnx`).

In [ ]:
VOICE = "suraj"          # same name as in the web app
LANGS = "hi mr en"       # languages that have a dataset in the zip
STEPS = 3000             # fine-tune steps per language

In [ ]:
# Install Piper's trainer and build its small C extension
!pip -q install "piper-tts[train]==1.8.0" torchaudio huggingface_hub
!git clone -q --depth 1 https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
import piper, os, glob, subprocess
ma = os.path.join(os.path.dirname(piper.__file__), 'train/vits/monotonic_align')
!cp /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx {ma}/
!cd {ma} && mkdir -p monotonic_align && cythonize -q -i core.pyx && mv core*.so monotonic_align/
from piper.train.vits.monotonic_align import maximum_path; print('trainer ready')

In [ ]:
# Upload distill/colab_<voice>.zip from the Mac
from google.colab import files
up = files.upload()
!mkdir -p /content/work && cd /content/work && unzip -qo /content/{next(iter(up))}
!ls /content/work/distill/data/{VOICE}

In [ ]:
# Fine-tune each language and export ONNX (about 20-40 min per language on a T4)
!cd /content/work && python distill/train_voice.py --voice {VOICE} --langs {LANGS} --steps {STEPS} --accelerator cuda --batch 32

In [ ]:
# Download the trained voices
!cd /content/work && zip -qr /content/piper_{VOICE}.zip voices/{VOICE}/piper
files.download(f'/content/piper_{VOICE}.zip')